# training-step-cycle — worked example 2: Verify that zero_grad truly clears gradients between steps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `training-step-cycle`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Forgetting `optimizer.zero_grad()` causes gradients to accumulate across steps: each backward call adds to existing `.grad` tensors rather than overwriting them. After two steps without zeroing, `param.grad` holds the sum of the gradients from both steps, causing the optimizer to take a much larger update than intended. This worked example demonstrates the accumulation bug by comparing the gradient after one step versus after two steps without zeroing.

## Worked solution

**Step 1 – Build a simple leaf and do ONE proper step.** We run the 5-call cycle once (including `zero_grad`). We snapshot `grad_after_proper` = the gradient computed in that step.

**Step 2 – Replicate but skip `zero_grad` for step two.** After the first proper step, we do a SECOND backward without zeroing. The gradient stored in `w.grad` is now the SUM of the two step gradients.

**Step 3 – Compare.** `grad_accumulated = w.grad.item()` will be larger in magnitude than `grad_after_proper` because it holds two steps worth of gradient. The ratio tells us how much accumulation occurred.

**Step 4 – Reset properly.** After demonstrating the bug we call `optimizer.zero_grad()` and verify `w.grad` is now zero, confirming that zeroing works.

In [ ]:
import torch as t

def worked2_zero_grad_demo():
    """
    Show that skipping zero_grad accumulates gradients.
    Returns dict with gradient measurements.
    """
    t.manual_seed(5)
    # Setup
    w = t.tensor([1.5], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=0.1)
    x = t.tensor([2.0, -1.0, 3.0])
    y_target = t.tensor([4.0, -2.0, 6.0])

    # === Proper step (with zero_grad) ===
    pred = w * x
    loss = ((pred - y_target) ** 2).mean()
    loss.backward()
    grad_step1 = w.grad.item()  # gradient from step 1 only
    optimizer.step()
    optimizer.zero_grad()       # clears the gradient
    grad_after_zero = w.grad    # should be None or tensor([0.])

    # === Bug: second backward WITHOUT zero_grad ===
    # This accumulates on top of whatever is in w.grad after the zero
    # (which is None, so first accumulation lands normally, but then if
    # we add a THIRD call without zero, it would stack)
    pred2 = w * x
    loss2 = ((pred2 - y_target) ** 2).mean()
    loss2.backward()  # Step A: this is now w.grad
    grad_after_step2_a = w.grad.item()
    pred3 = w * x  # fresh forward builds a NEW graph for the second backward
    loss3 = ((pred3 - y_target) ** 2).mean()
    loss3.backward()  # Step B: accumulate WITHOUT zero_grad — doubles the gradient
    grad_accumulated = w.grad.item()

    # Clean up
    optimizer.zero_grad()
    grad_after_cleanup = w.grad

    return {
        'grad_step1': grad_step1,
        'grad_after_step2_a': grad_after_step2_a,
        'grad_accumulated': grad_accumulated,
        'accumulation_ratio': grad_accumulated / grad_after_step2_a if grad_after_step2_a != 0 else None,
    }

result = worked2_zero_grad_demo()
print('grad step1:', f"{result['grad_step1']:.4f}")
print('grad after one step2 backward:', f"{result['grad_after_step2_a']:.4f}")
print('grad after accumulated (2x backward):', f"{result['grad_accumulated']:.4f}")
print('ratio (should be ~2.0):', f"{result['accumulation_ratio']:.4f}")